# LoRA fine-tuning of Bolmo 1B for low-resource IGT (Ainu, Tsez, Jejueo)

One notebook for everyone: set `LANG` and `STRATEGY` in section 3, the rest stays the same.
Agenda 24 Nov / 01 Dec: train **all-in-one** and **step-by-step** for your language (r16, lr 2e-4; no hyperparameter tuning now).
Both use the zero-shot prompt files in `zero_shot/prompts/`; test predictions can also use the few-shot prompt (`--shots K`).
Logic: `lora/lora_bolmo.py`. Settings: `configs/lora.json` (LoRA) and `configs/zero_shot.json` (languages, prompts, few-shot). Details: `lora/README.md`.

**Before running:** Runtime -> Change runtime type -> GPU (T4 works; L4 / A100 are faster and use bf16).
Everything a run produces goes to **Google Drive** (`MyDrive/igt-lora/<lang>/<run_id>/`), so if Colab disconnects, run the notebook again from the top and training continues from the last checkpoint.

In [ ]:
!nvidia-smi -L
import sys
print(sys.version)
assert sys.version_info >= (3, 11), "Bolmo needs xlstm, which needs Python >= 3.11"

## 1. Google Drive (checkpoints and results are saved here)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Get the code

In [ ]:
import os
REPO = "/content/Korean-IGT-Bolmo"
BRANCH = "theo/lora"   # until it is merged into main
ZIP = "/content/Korean-IGT-Bolmo.zip"   # optional: upload this zip in the Files panel instead of cloning
if os.path.exists(ZIP):
    !unzip -oq {ZIP} -d /content   # -o: a newly uploaded zip replaces the old code
elif not os.path.isdir(REPO):
    !git clone -b {BRANCH} https://github.com/JingyiShao-Gina/Korean-IGT-Bolmo.git {REPO}
else:
    !git -C {REPO} fetch -q origin {BRANCH} && git -C {REPO} checkout -q {BRANCH} && git -C {REPO} reset -q --hard origin/{BRANCH}
os.chdir(REPO)
!git log --oneline -1

Install dependencies. If transformers was already imported in this session, restart the session after this cell (Runtime -> Restart session) and run again from the top.

In [ ]:
# Bolmo was tested with transformers 4.57.3; Colab ships 5.x, which breaks its custom code.
!pip install -q "transformers==4.57.3" "xlstm==2.0.4" peft sacrebleu datasets pandas accelerate
# Colab preinstalls an old torchao; peft refuses to import next to it. This project does not use torchao.
!pip uninstall -y -q torchao

import importlib.util
missing = [m for m in ("xlstm", "mlstm_kernels", "peft", "sacrebleu") if importlib.util.find_spec(m) is None]
assert not missing, f"Not installed: {missing} - read the pip output above"
import transformers
print("transformers", transformers.__version__)
if transformers.__version__ != "4.57.3":
    print(">>> Restart the session (Runtime -> Restart session), then run the notebook again from the top.")

## 3. Choose the run

- `LANG`: `"ainu"`, `"tsez"` or `"jejueo"` (keys of `languages` in `configs/zero_shot.json`).
- `STRATEGY`: `"step_by_step"` or `"all_in_one"`. Train both, step-by-step first.
- `RUN`: `"baseline"` (r16, alpha 32, lr 2e-4). `r8` / `r32` / `lr1e-4` exist but are not planned now.
- `SEED`: 42.

In [ ]:
import os, sys, json
os.chdir("/content/Korean-IGT-Bolmo")  # a session restart resets the folder
sys.path.insert(0, "/content/Korean-IGT-Bolmo/lora")
import importlib, lora_bolmo as lb
importlib.reload(lb)

LANG = "ainu"
STRATEGY = "step_by_step"
RUN = "baseline"
SEED = 42

ARGS = ["--lang", LANG, "--strategy", STRATEGY, "--run", RUN, "--seed", str(SEED)]
cfg = lb.load_config()
s = lb.run_settings(cfg, RUN, STRATEGY, LANG, SEED)
print("run_id:", s["run_id"])
print("adapter:", s["adapter"])
print("training:", {k: s["training"][k] for k in ("learning_rate", "num_train_epochs", "gradient_accumulation_steps", "eval_every_epochs", "max_seq_bytes", "seed")})
print("prompt:\n" + lb.load_templates(LANG)[STRATEGY])

## 4. Data check (no GPU needed)

Rows per split, rows left out of training (empty field, or longer than `max_seq_bytes`), longest gold answer vs the generation budget, and the length of the 3-shot prompt.

In [ ]:
import check_data
importlib.reload(check_data)
check_data.main(["--lang", LANG, "--shots", "3"])

## 5. Smoke test (2 optimizer steps on 8 examples, 4 dev rows)

Checks that the model loads, LoRA attaches to the 32 global-transformer `q_proj`/`v_proj` modules, the loss runs, a checkpoint saves, and generation + chrF++ work, then runs test prediction with the zero-shot and the 3-shot prompt on 4 rows. Output goes to a separate `..._smoke` folder. **Not a result.**

In [ ]:
import pandas as pd
smoke_dir = lb.main(["train", *ARGS, "--smoke"])
print(json.load(open(smoke_dir / "run_meta.json"))["trainable_parameters"], "trainable parameters")
display(pd.read_csv(smoke_dir / "metrics.csv"))
display(pd.read_csv(sorted((smoke_dir / "dev_predictions").glob("*.csv"))[-1])[["id", "gold_transliteration", "pred_transliteration", "gold_gloss", "pred_gloss", "parse_status"]])
for shots in (0, 3):
    for p in lb.main(["predict", *ARGS, "--smoke", "--rule", "loss", "--shots", str(shots)]):
        print(p.name, pd.read_csv(p).parse_status.value_counts().to_dict())

## 6. Train (resumable)

Saves a checkpoint every 0.5 epoch, and for each checkpoint computes the teacher-forced **validation loss** and the **validation chrF++** of the generated gloss and transliteration (all dev rows). Progress is printed every 10 optimizer steps.
If Colab disconnects: run sections 1-3 again, then this cell - it continues from the last checkpoint or quick resume save (at most 20 optimizer steps are lost).

**Do not reload the Colab page while this cell runs**: the cell keeps running, but its output can no longer be shown.

In [ ]:
run_dir = lb.main(["train", *ARGS])

## 7. Validation loss vs chrF++ per checkpoint

In [ ]:
import matplotlib.pyplot as plt
run_dir = lb.default_out_root(cfg) / LANG / s["run_id"]
m = pd.read_csv(run_dir / "metrics.csv")
display(m)
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(m.epoch, m.val_loss, "o-", color="tab:blue"); ax1.set_xlabel("epoch"); ax1.set_ylabel("validation loss", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(m.epoch, m.val_gloss_chrfpp, "s-", color="tab:orange", label="gloss chrF++")
ax2.plot(m.epoch, m.val_transliteration_chrfpp, "^--", color="tab:green", label="transliteration chrF++")
ax2.set_ylabel("validation chrF++"); ax2.legend(loc="lower right")
plt.title(s["run_id"]); plt.tight_layout(); plt.show()

## 8. Select checkpoints and predict the test split

Selection A = lowest validation loss, selection B = highest validation gloss chrF++ (ties -> earlier step). Then test predictions for both, with the **zero-shot prompt** (main result, same prompt as training) and, as an extra comparison, the **3-shot prompt** (same examples as the few-shot experiment).
Only do this once the run is finished; never change settings after looking at test outputs.

In [ ]:
sel = lb.main(["select", *ARGS])
for shots in (0, 3):
    for p in lb.main(["predict", *ARGS, "--split", "test", "--rule", "both", "--shots", str(shots)]):
        df = pd.read_csv(p)
        print(p.name, df.parse_status.value_counts().to_dict(), "truncated:", int(df.truncated.sum()))

## 9. Save the small result files into the repo

Copies `run_meta.json`, `train_log.csv`, `metrics.csv`, `selection.json` and the test predictions (no adapters) to `results/lora/<lang>/<run_id>/`, then zips them for download. Adapters stay on Drive.

In [ ]:
dst = lb.main(["export", *ARGS])
RID = s["run_id"]
!cd /content/Korean-IGT-Bolmo && zip -qr /content/{RID}.zip results/lora/{LANG}/{RID}
from google.colab import files
files.download(f"/content/{RID}.zip")